# Notebook 4: Level 2 - Streaming Features + Hybrid Search

**What changes:**
- Add `claim_status` FeatureView backed by `PushSource` for real-time streaming
- Use **hybrid search** (vector + full-text) via `retrieve_online_documents_v2`
- Combine batch profile + streaming claim status in the LLM prompt

**Key Feast concepts:**
- `PushSource`: External systems (Kafka, claim pipeline) push feature updates in real-time
- `store.push()`: Write streaming features to the online store
- Hybrid search: pgvector cosine similarity + PostgreSQL tsvector full-text ranking

In [ ]:
import os, sys
REPO_DIR = os.path.expanduser('~/life-insurance-rag')
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'app'))

from feast import FeatureStore
from sentence_transformers import SentenceTransformer
import pandas as pd
from datetime import datetime

store = FeatureStore(repo_path='feature_repo')
model = SentenceTransformer('all-MiniLM-L6-v2')

## Step 1: Simulate a Streaming Claim Update

In production, a claims processing pipeline pushes updates via Kafka or the Feast push API.
Here we simulate it with `store.push()`.

In [ ]:
# Simulate: Priya's claim just moved from 'docs_received' to 'approved'
claim_update = pd.DataFrame({
    'customer_id': ['POL-1001'],
    'claim_number': ['CLM-9087'],
    'claim_stage': ['approved'],
    'claim_filed_date': ['2024-10-10'],
    'last_update': [datetime.now().strftime('%Y-%m-%d %H:%M')],
    'documents_pending': [0],
    'estimated_days_remaining': [3],
    'event_timestamp': [datetime.now()],
})

store.push('claim_status_push', claim_update)
print('Pushed claim update: stage=approved, docs_pending=0, est_days=3')

## Step 2: Fetch Batch + Streaming Features Together

In [ ]:
customer_id = 'POL-1001'

features = store.get_online_features(
    features=[
        # Batch
        'customer_profile:customer_name',
        'customer_profile:plan_type',
        'customer_profile:coverage_amount',
        # Streaming
        'claim_status:claim_number',
        'claim_status:claim_stage',
        'claim_status:documents_pending',
        'claim_status:estimated_days_remaining',
        'claim_status:last_update',
    ],
    entity_rows=[{'customer_id': customer_id}],
).to_dict()

print('--- Combined Batch + Streaming Features ---')
for k, v in features.items():
    if k != 'customer_id':
        print(f'  {k}: {v[0]}')

## Step 3: Hybrid Search (Vector + Full-Text)

Feast's hybrid search combines:
- **Vector similarity** (pgvector cosine distance) for semantic matching
- **Full-text search** (PostgreSQL tsvector/ts_rank) for keyword matching

This improves results when the query contains specific terms like claim numbers.

In [ ]:
query = 'What is happening with my claim?'
query_vector = model.encode(query).tolist()

# Standard vector search
vector_results = store.retrieve_online_documents(
    feature='insurance_docs:embedding',
    query=query_vector,
    top_k=3,
)
vector_df = vector_results.to_df()

print('=== Vector-Only Results ===')
for i, row in vector_df.iterrows():
    print(f'  [{i+1}] {row["title"]} ({row["category"]})')

In [ ]:
# Hybrid search (vector + full-text)
try:
    hybrid_results = store.retrieve_online_documents_v2(
        feature='insurance_docs:embedding',
        query=query_vector,
        native_query=query,
        top_k=3,
    )
    hybrid_df = hybrid_results.to_df()
    print('=== Hybrid Search Results ===')
    for i, row in hybrid_df.iterrows():
        print(f'  [{i+1}] {row["title"]} ({row["category"]})')
except Exception as e:
    print(f'Hybrid search not available: {e}')
    print('Using vector-only results as fallback.')
    hybrid_df = vector_df

## Step 4: Build Prompt with Streaming Context

In [ ]:
name = features['customer_name'][0]
claim_stage = features['claim_stage'][0]
days_left = features['estimated_days_remaining'][0]

doc_context = '\n---\n'.join(
    f'{row["title"]}: {row["content"]}'
    for _, row in hybrid_df.iterrows()
)

prompt = f"""You are a helpful life insurance assistant with real-time data.

CUSTOMER: {name}
  Plan: {features['plan_type'][0]}
  Claim: #{features['claim_number'][0]} - Stage: {claim_stage}
  Filed: {features['claim_filed_date'][0]}
  Est. days remaining: {days_left}
  Docs pending: {features['documents_pending'][0]}

DOCUMENTS:
{doc_context}

QUESTION: {query}

Answer with specific details about the customer's claim."""

print('--- Prompt includes LIVE claim status ---')
print(f'Claim stage: {claim_stage} (just updated via push!)')
print(f'Search type: hybrid (vector + full-text)')

## Key Insight

The answer now includes **real-time claim status** that was just pushed moments ago.
The customer sees "your claim has been approved" instead of outdated information.

---
**Next:** Open `05_level3_realtime_full.ipynb` for the complete picture.